In [1]:
# ============================================================
# PHASE 3 - Star Schema Design
# ============================================================

import pandas as pd
import numpy as np
import os
import warnings
warnings.filterwarnings('ignore')

BASE_DIR   = r"C:\Users\yipch\ecommerce-analytics-portfolio"
CLEAN_DIR  = os.path.join(BASE_DIR, "data", "processed")
EXPORT_DIR = os.path.join(BASE_DIR, "data", "exports")

# Load clean dataset from Phase 2
df = pd.read_csv(
    os.path.join(CLEAN_DIR, "clean_events.csv"),
    parse_dates=["event_time", "event_date"]
)

print(f"✅ Loaded clean dataset")
print(f"   Shape   : {df.shape[0]:,} rows × {df.shape[1]} cols")
print(f"   Columns : {list(df.columns)}")

✅ Loaded clean dataset
   Shape   : 1,605,102 rows × 22 cols
   Columns : ['event_time', 'event_type', 'product_id', 'category_id', 'category_code', 'brand', 'price', 'user_id', 'user_session', 'month', 'event_date', 'event_hour', 'event_dow', 'event_dow_num', 'event_week', 'event_month', 'category_l1', 'category_l2', 'revenue', 'session_converted', 'session_event_count', 'user_has_purchased']


In [2]:
# ============================================================
# STEP 2 - Star Schema Blueprint
# ============================================================

blueprint = """
╔══════════════════════════════════════════════════════════╗
║              STAR SCHEMA BLUEPRINT                       ║
╠══════════════════════════════════════════════════════════╣
║                                                          ║
║   dim_date          dim_products       dim_users         ║
║   ─────────         ────────────       ─────────         ║
║   date_id (PK)      product_id (PK)    user_id (PK)      ║
║   full_date         category_id        user_has_purchased║
║   year              category_code      total_sessions    ║
║   month             category_l1        total_events      ║
║   week              category_l2                          ║
║   day_of_week       brand                                ║
║   hour              avg_price                            ║
║   is_weekend                                             ║
║                          │                               ║
║              ╔═══════════╧══════════╗                    ║
║              ║     fact_events      ║                    ║
║              ╠══════════════════════╣                    ║
║              ║  event_id   (PK)     ║                    ║
║              ║  date_id    (FK) ────╫──→ dim_date        ║
║              ║  product_id (FK) ────╫──→ dim_products    ║
║              ║  user_id    (FK) ────╫──→ dim_users        ║
║              ║  session_id (FK) ────╫──→ dim_sessions    ║
║              ║  event_type          ║                    ║
║              ║  price               ║                    ║
║              ║  revenue             ║                    ║
║              ║  month               ║                    ║
║              ╚══════════════════════╝                    ║
║                          │                               ║
║                     dim_sessions                         ║
║                     ────────────                         ║
║                     session_id (PK)                      ║
║                     user_id                              ║
║                     session_converted                    ║
║                     session_event_count                  ║
╚══════════════════════════════════════════════════════════╝
"""
print(blueprint)


╔══════════════════════════════════════════════════════════╗
║              STAR SCHEMA BLUEPRINT                       ║
╠══════════════════════════════════════════════════════════╣
║                                                          ║
║   dim_date          dim_products       dim_users         ║
║   ─────────         ────────────       ─────────         ║
║   date_id (PK)      product_id (PK)    user_id (PK)      ║
║   full_date         category_id        user_has_purchased║
║   year              category_code      total_sessions    ║
║   month             category_l1        total_events      ║
║   week              category_l2                          ║
║   day_of_week       brand                                ║
║   hour              avg_price                            ║
║   is_weekend                                             ║
║                          │                               ║
║              ╔═══════════╧══════════╗                    ║
║              ║     fa

In [3]:
# ============================================================
# STEP 3 - dim_date
# One row per unique hour (finest granularity in our data)
# ============================================================

dim_date = (
    df[["event_time", "event_date", "event_hour",
        "event_dow", "event_dow_num", "event_week", "event_month"]]
    .drop_duplicates(subset=["event_time"])
    .copy()
)

# Rename for clarity
dim_date = dim_date.rename(columns={
    "event_time"    : "full_datetime",
    "event_date"    : "full_date",
    "event_hour"    : "hour",
    "event_dow"     : "day_of_week",
    "event_dow_num" : "day_of_week_num",
    "event_week"    : "week_number",
    "event_month"   : "year_month",
})

# Add extra date fields
dim_date["year"]       = pd.to_datetime(dim_date["full_date"]).dt.year
dim_date["month_num"]  = pd.to_datetime(dim_date["full_date"]).dt.month
dim_date["day"]        = pd.to_datetime(dim_date["full_date"]).dt.day
dim_date["is_weekend"] = dim_date["day_of_week_num"].isin([5, 6]).astype(int)

# Surrogate key
dim_date = dim_date.reset_index(drop=True)
dim_date.insert(0, "date_id", dim_date.index + 1)

print(f"✅ dim_date built")
print(f"   Rows    : {dim_date.shape[0]:,}")
print(f"   Columns : {list(dim_date.columns)}")
print(f"\n   Sample:")
print(dim_date.head(3).to_string())

✅ dim_date built
   Rows    : 1,288,421
   Columns : ['date_id', 'full_datetime', 'full_date', 'hour', 'day_of_week', 'day_of_week_num', 'week_number', 'year_month', 'year', 'month_num', 'day', 'is_weekend']

   Sample:
   date_id             full_datetime  full_date  hour day_of_week  day_of_week_num  week_number year_month  year  month_num  day  is_weekend
0        1 2019-10-15 08:02:19+00:00 2019-10-15     8     Tuesday                1           42    2019-10  2019         10   15           0
1        2 2019-10-15 08:02:28+00:00 2019-10-15     8     Tuesday                1           42    2019-10  2019         10   15           0
2        3 2019-10-15 08:03:21+00:00 2019-10-15     8     Tuesday                1           42    2019-10  2019         10   15           0


In [4]:
# ============================================================
# STEP 4 - dim_products
# One row per unique product
# ============================================================

dim_products = (
    df.groupby("product_id")
    .agg(
        category_id   = ("category_id",   "first"),
        category_code = ("category_code", "first"),
        category_l1   = ("category_l1",   "first"),
        category_l2   = ("category_l2",   "first"),
        brand         = ("brand",         "first"),
        avg_price     = ("price",         "mean"),
        min_price     = ("price",         "min"),
        max_price     = ("price",         "max"),
        times_viewed  = ("event_type",    lambda x: (x == "view").sum()),
        times_carted  = ("event_type",    lambda x: (x == "cart").sum()),
        times_bought  = ("event_type",    lambda x: (x == "purchase").sum()),
    )
    .reset_index()
)

dim_products["avg_price"] = dim_products["avg_price"].round(2)

print(f"✅ dim_products built")
print(f"   Rows    : {dim_products.shape[0]:,}")
print(f"   Columns : {list(dim_products.columns)}")
print(f"\n   Top 5 most purchased products:")
print(dim_products.nlargest(5, "times_bought")[
    ["product_id","brand","category_l1","avg_price","times_bought"]
].to_string())

✅ dim_products built
   Rows    : 95,684
   Columns : ['product_id', 'category_id', 'category_code', 'category_l1', 'category_l2', 'brand', 'avg_price', 'min_price', 'max_price', 'times_viewed', 'times_carted', 'times_bought']

   Top 5 most purchased products:
       product_id    brand  category_l1  avg_price  times_bought
783       1004856  samsung  electronics     129.21           871
709       1004767  samsung  electronics     247.44           647
977       1005115    apple  electronics     949.18           505
12615     4804056    apple  electronics     161.94           442
762       1004833  samsung  electronics     171.42           360


In [5]:
# ============================================================
# STEP 5 - dim_users
# One row per unique user
# ============================================================

dim_users = (
    df.groupby("user_id")
    .agg(
        user_has_purchased  = ("user_has_purchased", "max"),
        total_events        = ("event_type",          "count"),
        total_views         = ("event_type",          lambda x: (x == "view").sum()),
        total_carts         = ("event_type",          lambda x: (x == "cart").sum()),
        total_purchases     = ("event_type",          lambda x: (x == "purchase").sum()),
        total_sessions      = ("user_session",        "nunique"),
        total_revenue       = ("revenue",             "sum"),
        first_seen          = ("event_time",          "min"),
        last_seen           = ("event_time",          "max"),
        active_months       = ("event_month",         "nunique"),
    )
    .reset_index()
)

dim_users["total_revenue"]    = dim_users["total_revenue"].round(2)
dim_users["avg_order_value"]  = np.where(
    dim_users["total_purchases"] > 0,
    (dim_users["total_revenue"] / dim_users["total_purchases"]).round(2),
    0
)

print(f"✅ dim_users built")
print(f"   Rows    : {dim_users.shape[0]:,}")
print(f"   Columns : {list(dim_users.columns)}")
print(f"\n   Buyer vs Non-buyer breakdown:")
buyers = dim_users["user_has_purchased"].value_counts()
print(f"   Buyers (1)     : {buyers.get(1,0):,}")
print(f"   Non-buyers (0) : {buyers.get(0,0):,}")

✅ dim_users built
   Rows    : 99,658
   Columns : ['user_id', 'user_has_purchased', 'total_events', 'total_views', 'total_carts', 'total_purchases', 'total_sessions', 'total_revenue', 'first_seen', 'last_seen', 'active_months', 'avg_order_value']

   Buyer vs Non-buyer breakdown:
   Buyers (1)     : 11,698
   Non-buyers (0) : 87,960


In [6]:
# ============================================================
# STEP 6 - dim_sessions
# One row per unique session
# ============================================================

dim_sessions = (
    df.groupby("user_session")
    .agg(
        user_id              = ("user_id",              "first"),
        session_converted    = ("session_converted",    "max"),
        session_event_count  = ("session_event_count",  "first"),
        session_start        = ("event_time",           "min"),
        session_end          = ("event_time",           "max"),
        unique_products      = ("product_id",           "nunique"),
        session_revenue      = ("revenue",              "sum"),
    )
    .reset_index()
    .rename(columns={"user_session": "session_id"})
)

dim_sessions["session_duration_min"] = (
    (dim_sessions["session_end"] - dim_sessions["session_start"])
    .dt.total_seconds() / 60
).round(2)

dim_sessions["session_revenue"] = dim_sessions["session_revenue"].round(2)

conv_rate = dim_sessions["session_converted"].mean()
avg_dur   = dim_sessions["session_duration_min"].mean()

print(f"✅ dim_sessions built")
print(f"   Rows              : {dim_sessions.shape[0]:,}")
print(f"   Columns           : {list(dim_sessions.columns)}")
print(f"\n   Session conversion rate  : {conv_rate:.2%}")
print(f"   Avg session duration     : {avg_dur:.1f} minutes")

✅ dim_sessions built
   Rows              : 341,526
   Columns           : ['session_id', 'user_id', 'session_converted', 'session_event_count', 'session_start', 'session_end', 'unique_products', 'session_revenue', 'session_duration_min']

   Session conversion rate  : 6.14%
   Avg session duration     : 17.5 minutes


In [7]:
# ============================================================
# STEP 7 - fact_events (the central fact table)
# Contains all measurable events + foreign keys to dimensions
# ============================================================

# Build a datetime-to-date_id lookup
datetime_to_id = dim_date.set_index("full_datetime")["date_id"].to_dict()

fact_events = df[[
    "event_time", "user_id", "product_id",
    "user_session", "event_type", "price",
    "revenue", "month"
]].copy().rename(columns={"user_session": "session_id"})

# Add surrogate key
fact_events.insert(0, "event_id", range(1, len(fact_events) + 1))

# Add foreign key: date_id
fact_events["date_id"] = fact_events["event_time"].map(datetime_to_id)

# Drop raw event_time (now linked via date_id → dim_date)
fact_events = fact_events.drop(columns=["event_time"])

# Final column order
fact_events = fact_events[[
    "event_id", "date_id", "user_id",
    "product_id", "session_id", "event_type",
    "price", "revenue", "month"
]]

print(f"✅ fact_events built")
print(f"   Rows    : {fact_events.shape[0]:,}")
print(f"   Columns : {list(fact_events.columns)}")
print(f"\n   Sample:")
print(fact_events.head(5).to_string())

# Quick integrity check
print(f"\n🔍 Integrity Checks:")
print(f"   Null date_id     : {fact_events['date_id'].isnull().sum():,}")
print(f"   Null user_id     : {fact_events['user_id'].isnull().sum():,}")
print(f"   Null product_id  : {fact_events['product_id'].isnull().sum():,}")
print(f"   Null session_id  : {fact_events['session_id'].isnull().sum():,}")

✅ fact_events built
   Rows    : 1,605,102
   Columns : ['event_id', 'date_id', 'user_id', 'product_id', 'session_id', 'event_type', 'price', 'revenue', 'month']

   Sample:
   event_id  date_id    user_id  product_id                            session_id event_type   price  revenue     month
0         1        1  520649833     3600661  00005026-a9d1-4e2b-8290-3cc14e4bad89       view  296.02      0.0  2019-Oct
1         2        2  520649833     3600163  00005026-a9d1-4e2b-8290-3cc14e4bad89       view  182.41      0.0  2019-Oct
2         3        3  520649833     3600163  00005026-a9d1-4e2b-8290-3cc14e4bad89       cart  182.41      0.0  2019-Oct
3         4        4  520649833     3600163  00005026-a9d1-4e2b-8290-3cc14e4bad89       view  182.41      0.0  2019-Oct
4         5        5  520649833     2500566  00005026-a9d1-4e2b-8290-3cc14e4bad89       view   61.22      0.0  2019-Oct

🔍 Integrity Checks:
   Null date_id     : 0
   Null user_id     : 0
   Null product_id  : 0
   Null sessi

In [8]:
# ============================================================
# STEP 8 - Summary + Save all tables as CSV
# ============================================================

tables = {
    "fact_events"   : fact_events,
    "dim_date"      : dim_date,
    "dim_products"  : dim_products,
    "dim_users"     : dim_users,
    "dim_sessions"  : dim_sessions,
}

print("💾 Saving all star schema tables...\n")
print(f"{'Table':<20} {'Rows':>10} {'Cols':>6} {'Size (MB)':>10}")
print("-" * 50)

for name, table in tables.items():
    path = os.path.join(EXPORT_DIR, f"{name}.csv")
    table.to_csv(path, index=False)
    size_mb = os.path.getsize(path) / (1024 * 1024)
    print(f"{name:<20} {table.shape[0]:>10,} {table.shape[1]:>6} {size_mb:>9.1f}MB")

print("\n✅ All tables saved to data/exports/")
print("\n🎯 These files go directly into MySQL (Phase 4) and Power BI (Phase 8)")

💾 Saving all star schema tables...

Table                      Rows   Cols  Size (MB)
--------------------------------------------------
fact_events           1,605,102      9     146.2MB
dim_date              1,288,421     12     100.3MB
dim_products             95,684     12       8.6MB
dim_users                99,658     12       8.2MB
dim_sessions            341,526      9      37.5MB

✅ All tables saved to data/exports/

🎯 These files go directly into MySQL (Phase 4) and Power BI (Phase 8)
